# ATML PA4 Task 3 full pipeline

Smoke, baseline, distill, merge, eval, analyze.

In [ ]:
import gzip, os, shutil, subprocess, sys
from pathlib import Path

root = Path('/kaggle/input')
print('inputs', [p.name for p in root.iterdir()], flush=True)
locked = next(root.rglob('c4_calibration_locked.jsonl*'))
source = locked.parents[2]
data = Path('/kaggle/working/pa4-data')
if data.exists():
    shutil.rmtree(data)
gzip_dirs = {'public_locked', 'kv_retrieval', 'mqar'}
for path in source.rglob('*'):
    if not path.is_file():
        continue
    rel = path.relative_to(source)
    if path.suffix == '.jsonl' and path.parent.name in gzip_dirs:
        target = data / rel.with_suffix('.jsonl.gz')
        target.parent.mkdir(parents=True, exist_ok=True)
        with path.open('rb') as raw, gzip.open(target, 'wb') as packed:
            shutil.copyfileobj(raw, packed)
    else:
        target = data / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
print('data', data, flush=True)

repo = Path('/kaggle/working/ATML-PA4')
if repo.exists():
    shutil.rmtree(repo)
subprocess.check_call(['git', 'clone', 'https://github.com/ttqureshi/ATML-PA4.git', str(repo)])
subprocess.check_call(['git', '-C', str(repo), 'checkout', '--detach', 'origin/task3'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.1', 'huggingface_hub==0.36.2', 'accelerate', 'peft', 'matplotlib', 'sentencepiece', 'protobuf'])

env = os.environ.copy()
env['PA4_DATA_REPO'] = str(data)
env['PYTHONUNBUFFERED'] = '1'
env['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

for stage in ['smoke', 'baseline', 'distill', 'merge', 'eval', 'analyze']:
    print('\n====', stage, '====', flush=True)
    rc = subprocess.run([sys.executable, 'task3/run.py', stage], cwd=repo, env=env).returncode
    print('exit', rc, flush=True)
    if rc != 0:
        raise SystemExit(rc)

bundle = Path('/kaggle/working/task3_results')
if bundle.exists():
    shutil.rmtree(bundle)
src = repo / 'results' / 'task3'
if not src.exists():
    # some layouts write under task3/results
    alt = repo / 'task3' / 'results'
    src = alt if alt.exists() else src
if src.exists():
    shutil.copytree(src, bundle)
print('TASK3 COMPLETE', flush=True)
